# Milestone 2 lab, notebook 8 of 8: The production version

## Where we are

| Notebook | What you do |
|---|---|
| 1. Empty in which rows | Write a check for empty cells that does not fire on good rows |
| 2. Unique on which columns | Find the receipt and the file that were counted twice |
| 3. A believable value | Catch 1,180 kg of bananas without catching 400 real bottles |
| 4. Rules of the business | Three rules a person must tell you. Each needs a join first |
| 5. Across two tables | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| 6. A normal day | Find the store that sent nothing. Work out what a normal day is |
| 7. The false alarm | A holiday is not an error. See what silence costs |
| **8. The production version** (you are here) | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running.

**About five minutes.**

In [1]:
import subprocess
import sys

from lab_helpers import MILESTONE_DIR, load_result, prepare_week, store_days, production
from lotus_pipeline import inspect, pipeline

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: From a lab to a program

In seven notebooks you wrote thirteen checks by hand. Each of them is also in the production code, in `lotus_pipeline/sales_checks.py`, with its reason written beside its query. Three more, for the tills' own totals, are in `till_checks.py`.

| You wrote, in notebook | The production checks |
|---|---|
| 1. Empty in which rows | `required_cells` |
| 2. Unique on which columns | `true_key_unique` |
| 3. A believable value | `believable_quantity`, `right_shape`, `line_adds_up` |
| 4. Rules of the business | `refund_gives_money_back`, `refund_not_larger_than_sale`, `price_of_the_day`, `inside_opening_hours` |
| 5. Across two tables | `till_total`, `every_sale_has_a_product` |
| 6 and 7. A normal day, and the false alarm | `every_store_reported`, `normal_volume` |

One command runs all sixteen. In a terminal, from this milestone's folder: `uv run python -m lotus_pipeline inspect`.

In [2]:
_ = pipeline.main(["inspect"])

The checks with a reason: 13 on sales_lines, 3 on till_totals

  Through the data lens
     1. A cell that must be filled is filled           pass
     2. One item line is stored once                   FAIL  171 bad rows   S01 08-25, S02 08-28
     3. A receipt id has the right shape               pass
     4. A line adds up                                 FAIL    1 bad rows   S03 08-24
     5. Every sale is for a product that exists        FAIL    3 bad rows   S06 08-29
    14. A till has one total for a day                 pass

  Through the business lens
     6. A refund gives money back                      FAIL    1 bad rows   S03 08-24
     7. A refund is not larger than the sale           pass
     8. The price charged is the price valid that day  FAIL    6 bad rows   S04 08-28
     9. Nothing is sold while the store is closed      FAIL    4 bad rows   S02 08-31
    10. A weight is a believable weight                FAIL    1 bad rows   S05 08-27
    11. The receipts add up to 

Read it by lens.

- **Data:** the values themselves. Is a cell empty, is a line there twice, does a line add up?
- **Business:** a rule of how Lotus works. A refund gives money back. A till's money equals its receipts.
- **Operational:** the pipeline. Did every store send a file, with a normal amount of data?

Ten of the sixteen found bad rows. In milestone 1, all five checks were green on the same data.

To see one check with its reason and its rows:

In [3]:
_ = pipeline.main(["inspect", "--check", "till_total"])

11. The receipts add up to the till's own total
   lens:      business
   answers:   Accurate?
   protects:  Store managers and the head of retail operations: net sales, so pay. It is the only check that can say a whole day's money is right
   prepare:   Add the lines up per till per day. Join that to till_totals, the till's own count of the same money
   limit:     The two totals are equal, to the cent. A till with no lines, and lines with no till, both fail

   11 bad rows:
   store_id=S03  till_id=T1  sales_date=2026-08-24  till_cents=41959  lines_cents=43857  difference_cents=1898
   store_id=S01  till_id=T2  sales_date=2026-08-25  till_cents=37303  lines_cents=39110  difference_cents=1807
   store_id=S04  till_id=T1  sales_date=2026-08-25  till_cents=31289  lines_cents=0  difference_cents=-31289
   store_id=S04  till_id=T2  sales_date=2026-08-25  till_cents=27453  lines_cents=0  difference_cents=-27453
   store_id=S05  till_id=T1  sales_date=2026-08-27  till_cents=25234  lines_cen

## Step 2: Do your checks agree with the program?

Every time you ran `keep(...)`, your hand-written check was saved. Hold all thirteen against the production checks.

In [4]:
saved = load_result("answer_key_checks")
equal = 0
for key, yours in saved.items():
    theirs = production(key)
    same = yours["bad_rows"] == len(theirs) and yours["store_days"] == store_days(theirs)
    equal += same
    print(f"{key:<30} yours: {yours['bad_rows']:>3} bad rows   the program: {len(theirs):>3}   {'equal' if same else 'DIFFERENT'}")
print(f"\n{equal} of {len(saved)} hand-written checks equal the production checks.")

believable_quantity            yours:   1 bad rows   the program:   1   equal
every_sale_has_a_product       yours:   3 bad rows   the program:   3   equal
every_store_reported           yours:   1 bad rows   the program:   1   equal


inside_opening_hours           yours:   4 bad rows   the program:   4   equal
line_adds_up                   yours:   1 bad rows   the program:   1   equal
normal_volume                  yours:   1 bad rows   the program:   1   equal
price_of_the_day               yours:   6 bad rows   the program:   6   equal
refund_gives_money_back        yours:   1 bad rows   the program:   1   equal
refund_not_larger_than_sale    yours:   0 bad rows   the program:   0   equal


required_cells                 yours:   0 bad rows   the program:   0   equal


right_shape                    yours:   0 bad rows   the program:   0   equal


till_total                     yours:  11 bad rows   the program:  11   equal
true_key_unique                yours: 171 bad rows   the program: 171   equal

13 of 13 hand-written checks equal the production checks.


## Step 3: Which check catches which fault?

Nine faults were planted in the sales files of the bonus week. Each production check names the cases it catches.

In [5]:
names = {1: "A refund stored as a positive amount", 2: "One receipt in the file twice",
         3: "A weight in grams, not kilograms", 4: "Yesterday's price charged",
         5: "A sale at 03:10 in a closed store", 6: "A product that is not in products",
         7: "One store sends no file", 8: "One store's file a day late",
         9: "One store's whole file twice"}
for case, name in names.items():
    caught_by = [c.key for c in inspect.ALL_CHECKS if case in c.catches]
    print(f"{case}. {name:<38} {', '.join(caught_by)}")

1. A refund stored as a positive amount   line_adds_up, refund_gives_money_back, till_total
2. One receipt in the file twice          true_key_unique, till_total
3. A weight in grams, not kilograms       believable_quantity, till_total
4. Yesterday's price charged              price_of_the_day
5. A sale at 03:10 in a closed store      inside_opening_hours, till_total
6. A product that is not in products      every_sale_has_a_product, till_total
7. One store sends no file                till_total, every_store_reported
8. One store's file a day late            every_store_reported
9. One store's whole file twice           true_key_unique, till_total, normal_volume


Every case is caught by at least one check, and most by two. Two different checks on one fault is not waste: the till total says **that** money is wrong, and the business rule says **what** is wrong.

Case 8, the late file, is caught on the night it is missing and is gone the next night, when the file arrives. Telling "late" from "missing" needs to know **when** each file arrived. That is milestone 5.

## Step 4: The tests

The rule from milestone 1 still holds: **every check has two tests**. One where its fault is there and the check returns exactly the bad rows. One on four clean weeks where it stays silent. And none may fire on the holiday, the closed store or the 400 bottles.

In [6]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_sales_checks.py", "-q", "--no-header", "--color=no",
     "-p", "no:cacheprovider", "--collect-only"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
tests = sorted({line.split("::")[-1].split("[")[0] for line in done.stdout.splitlines() if "::" in line})
print("\n".join(tests))

test_a_day_that_is_far_too_small
test_a_refund_larger_than_the_sale
test_a_refund_of_a_receipt_that_does_not_exist
test_a_sale_before_the_holiday_opening_hour
test_a_till_that_sent_no_total
test_a_till_total_stored_twice
test_a_till_total_that_does_not_add_up
test_case_8_the_late_file_is_seen_on_its_night_and_gone_the_next
test_each_check_catches_exactly_its_planted_cases
test_every_check_can_say_why_it_exists
test_every_check_is_silent_on_four_clean_weeks
test_half_a_bottle
test_no_check_fires_on_the_things_that_look_wrong_and_are_not
test_required_cells_an_empty_amount
test_required_cells_empty_in_which_rows
test_right_shape
test_the_amounts_the_checks_report
test_the_catches_named_on_each_check_are_the_truth
test_the_five_standard_checks_cannot_say_why_they_exist
test_the_report_and_the_command_line
test_the_till_total_says_how_far_apart_the_two_counts_are


In [7]:
done = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_sales_checks.py", "-q", "--no-header", "--color=no",
     "-p", "no:cacheprovider"],
    cwd=MILESTONE_DIR, capture_output=True, text=True,
)
print(done.stdout.strip().splitlines()[-1])

45 passed in 6.94s


To run every test of the milestone, type `uv run pytest` in this folder.

## Step 5: What is still wrong

You now have sixteen checks with a reason, and they find every planted fault in the sales files. So is the bonus safe?

Ask the pipeline. This is the report the head of retail operations gets on Monday.

In [8]:
_ = pipeline.main(["report", "--day", "2026-08-30"])


Lotus Retails. Store managers' bonus, week of Mon 2026-08-24
Net sales from Mon 2026-08-24 to Sun 2026-08-30

  Store                           Manager              Net sales  Bonus (1%)
  S01  Lotus Ashby High Street    Marta Lindqvist      $5,171.32      $51.71
  S02  Lotus Ashby Market Square  Dana Okafor          $6,758.96      $67.59
  S03  Lotus Calder Bridge        Tomas Reyes          $5,106.60      $51.07
  S04  Lotus Calder Station       Ines Marchetti       $4,585.57      $45.86
  S05  Lotus Fenwick Green        Samir Haddad         $5,113.73      $51.14
  S06  Lotus Fenwick Harbour      Aiko Tanaka          $4,400.92      $44.01
  All six stores                                      $31,137.10     $311.38

  Checks: 5 of 5 passed.


The same wrong numbers, and the last line still says **5 of 5 passed**.

Your checks are not part of the nightly run. `inspect` only looks. It stops nothing, sets nothing aside, and tells nobody. That changes in milestone 4.

And there is a nearer problem. Lotus has seven tables. You have done `sales_lines`, with `till_totals` beside it as its control. The other five are a price list typed by people, six rows about stores, a stock count done by hand, a file from an outside company, and a list of members typed by customers. Almost nothing you wrote here fits them.

> **One table is done well. There are five more, and they are nothing like it.**

Milestone 3 gives each of them its own check card.

## Before you go on

- Open [milestone_2_flow.html](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m2_checks_with_a_reason/milestone_2_flow.html) in a browser. Pick a fault and watch which check stops it.
- Open [milestone_2_recap.html](https://vijaysngrajput.github.io/quality-gates-on-a-running-pipeline/src/m2_checks_with_a_reason/milestone_2_recap.html) for a look back, and three questions to test yourself.
- Stop what you started: `docker compose down`, from the project folder.